# 02 — Intent Discovery

Cluster SpotifyCare customer messages to discover natural intent groups.

**Method**: sentence-transformers embeddings → UMAP → HDBSCAN / k-means

**Output**: 9 labelled intents used in the classifier taxonomy.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load pre-computed data
df = pd.read_csv("../data/processed/clean.csv")
embeddings = np.load("../data/processed/embeddings.npy")
print(f"Loaded {len(df):,} messages with embeddings of shape {embeddings.shape}")


In [ ]:
# Reduce to 2D with UMAP for visualisation
import umap

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding_2d = reducer.fit_transform(embeddings)
print("UMAP reduction complete:", embedding_2d.shape)


In [ ]:
# Cluster with HDBSCAN
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=10)
labels = clusterer.fit_predict(embedding_2d)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct = (labels == -1).mean() * 100
print(f"HDBSCAN found {n_clusters} clusters, {noise_pct:.1f}% noise")


In [ ]:
# Visualise clusters
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    embedding_2d[:, 0], embedding_2d[:, 1],
    c=labels, cmap="tab20", s=2, alpha=0.6
)
ax.set_title(f"UMAP + HDBSCAN Clustering — {n_clusters} clusters")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.colorbar(scatter, ax=ax, label="Cluster")
plt.tight_layout()
plt.savefig("../report/intent_clusters.png", dpi=120)
plt.show()


In [ ]:
# Inspect top keywords per cluster to assign intent labels
from sklearn.feature_extraction.text import TfidfVectorizer

df["cluster"] = labels
INTENT_LABELS = {
    # Manually assigned after inspecting cluster keywords
    0: "playback_issue",
    1: "app_device_bug",
    2: "account_billing",
    3: "download_offline",
    4: "search_discovery",
    5: "content_availability",
    6: "social_playlist",
    7: "compliment",
    # -1 (noise) → "other"
}

for cluster_id in sorted(df["cluster"].unique()):
    if cluster_id == -1:
        continue
    cluster_msgs = df[df["cluster"] == cluster_id]["customer_message_clean"].dropna()
    if len(cluster_msgs) < 5:
        continue
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10, stop_words="english")
    vec.fit(cluster_msgs)
    top_words = ", ".join(vec.get_feature_names_out())
    label = INTENT_LABELS.get(cluster_id, f"cluster_{cluster_id}")
    print(f"[{cluster_id:>3}] {label:<25} | top terms: {top_words}")


In [ ]:
# Save cluster assignments to processed data for use in evaluation
df["predicted_intent_cluster"] = df["cluster"].map(INTENT_LABELS).fillna("other")
df.to_csv("../data/processed/clean.csv", index=False)
print("Saved intent cluster labels to clean.csv")
print()
print("Intent distribution:")
print(df["predicted_intent_cluster"].value_counts().to_string())
